In [1]:
# Pomysł utworzenia prostej sieci neuronowej, przyjmującej 5 prawdopodobieństw z warstw
# Zwraca jedno prawdopodobieństwo, które jest ostatecznym wynikiem, wraz z informacją "autentyczna/podstawiona"
import torch
import pandas as pd
import torch.nn as nn
import torch.utils.data

In [2]:
# Dane testowe w formie: (P1, P2, P3, P4, P5, wynik)
# P1 - wykrywanie ramek
# P2 - wykrywanie smartfona
# P3 - analiza kontekstu
# P4 - analiza oświetlenia
# P5 - sieć neuronowa
df = pd.read_csv('data.csv')

print(df)

In [3]:
# Prosta sieć neuronowa oceniająca prawdopodobieństwo podstawienia twarzy na podstawie 5 prawdopodobieństw częściowych
class ProbabilityNN(nn.Module):
    def __init__(self):
        super(ProbabilityNN, self).__init__()
        self.fc1 = nn.Linear(5, 10)
        self.fc2 = nn.Linear(10, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x
    
model = ProbabilityNN()

In [4]:
# Podzielenie zbioru na dane wejściowe i wyjściowe
inputs = df[['P1', 'P2', 'P3', 'P4', 'P5']]
outputs = df['wynik']

dataset = torch.utils.data.TensorDataset(torch.tensor(inputs.values, dtype=torch.float32), torch.tensor(outputs.values, dtype=torch.float32))
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 100

# Pętla treningowa
for epoch in range(num_epochs):
    for x, y in dataloader:
        y = y.view(-1, 1)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
    print(f'Epoch: {epoch}, Loss: {loss.item()}')

In [5]:
# Przetestowanie modelu
model.eval()
with torch.no_grad():
    sample_input = torch.tensor([[1.0, 0.0, 0.0, 0.0, 1.0]], dtype=torch.float32)
    prediction = model(sample_input)
    print(f'Prawdopodobieństwo: {prediction.item():.4f}')

In [6]:
# Zapisanie modelu
torch.save(model.state_dict(), 'probability_nn.pth')